In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/1

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules.

Deep Learning takes Machine Learning further by using neural networks with many layers to
automatically discover complex patterns in data.

Retrieval-Augmented Generation, or RAG, combines a search system with a language model so answers
are grounded in real retrieved documents instead of only the model's memorized training data.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching, allowing
systems to find results that match the meaning of a query, not just its exact words.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
""",
    "product_catalog.txt": """Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

The standing desk converter is listed under product code SKU-7788 and ships within three business
days.

Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220, chunk_overlap=30, separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_hybrid_demo", embedding_function=hf_embedder)

all_chunks = []       # flat list of chunk text, in the same order as ChromaDB ids
all_chunk_ids = []
all_chunk_meta = []

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    doc_ids = [f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    doc_meta = [{"source": source, "chunk_index": i} for i in range(len(doc_chunks))]

    collection.add(documents=doc_chunks, metadatas=doc_meta, ids=doc_ids)

    all_chunks.extend(doc_chunks)
    all_chunk_ids.extend(doc_ids)
    all_chunk_meta.extend(doc_meta)

print("Total chunks stored:", collection.count())
for cid, chunk in zip(all_chunk_ids, all_chunks):
    print(f"  {cid}: {chunk[:60]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 10
  ai_overview.txt_chunk_0: Artificial Intelligence: An Overview

Artificial Intelligenc...
  ai_overview.txt_chunk_1: Machine Learning is a subset of AI where systems learn patte...
  ai_overview.txt_chunk_2: Deep Learning takes Machine Learning further by using neural...
  ai_overview.txt_chunk_3: Retrieval-Augmented Generation, or RAG, combines a search sy...
  embeddings_in_practice.txt_chunk_0: Embeddings and Vector Search in Practice...
  embeddings_in_practice.txt_chunk_1: Modern search engines increasingly rely on embeddings rather...
  embeddings_in_practice.txt_chunk_2: Vector databases such as ChromaDB store these embeddings and...
  product_catalog.txt_chunk_0: Product Catalog Notes

The wireless noise-cancelling headpho...
  product_catalog.txt_chunk_1: The standing desk converter is listed under product code SKU...
  product_catalog.txt_chunk_2: Return requests for any product must reference the exact SKU...


In [3]:
def semantic_search(query, n_results=3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    output = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        output.append({"text": doc, "source": meta["source"], "similarity": round(1 - distance, 4)})
    return output

results = semantic_search("SKU-4521")
for r in results:
    print(f"[{r['similarity']}] ({r['source']}) {r['text']}")

[0.3853] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
[0.3851] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.2492] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.


In [4]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().replace("-", " ").split()

tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_search(query, n_results=3):
    scores = bm25.get_scores(tokenize(query))
    ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_results]

    output = []
    for idx in ranked_idx:
        if scores[idx] > 0:   # only keep chunks that actually matched a term
            output.append({
                "text": all_chunks[idx],
                "source": all_chunk_meta[idx]["source"],
                "bm25_score": round(float(scores[idx]), 4)
            })
    return output

results = bm25_search("SKU-4521")
for r in results:
    print(f"[{r['bm25_score']}] ({r['source']}) {r['text']}")

[2.5064] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.8352] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.
[0.7638] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.


In [5]:
results = bm25_search("a computer system that learns from experience")
print("BM25 results for a meaning-based query:")
if not results:
    print("  (no keyword matches found)")
for r in results:
    print(f"[{r['bm25_score']}] ({r['source']}) {r['text']}")

BM25 results for a meaning-based query:
[2.7926] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules.
[2.7265] (ai_overview.txt) Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.
[2.5515] (ai_overview.txt) Retrieval-Augmented Generation, or RAG, combines a search system with a language model so answers
are grounded in real retrieved documents instead of only the model's memorized training data.


In [6]:
def hybrid_search(query, n_results=3, k=60):
    semantic_results = semantic_search(query, n_results=10)
    keyword_results = bm25_search(query, n_results=10)

    rrf_scores = {}
    chunk_lookup = {}

    for rank, r in enumerate(semantic_results):
        key = r["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup[key] = r

    for rank, r in enumerate(keyword_results):
        key = r["text"]
        rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (k + rank + 1)
        chunk_lookup.setdefault(key, r)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:n_results]

    output = []
    for text, score in ranked:
        chunk_info = chunk_lookup[text]
        output.append({
            "text": text,
            "source": chunk_info["source"],
            "rrf_score": round(score, 5)
        })
    return output

results = hybrid_search("SKU-4521 product details")
for r in results:
    print(f"[{r['rrf_score']}] ({r['source']}) {r['text']}")

[0.03252] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.
[0.03227] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
[0.032] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.


In [7]:
import pandas as pd

test_query = "What is SKU-4521?"

semantic_top = semantic_search(test_query, n_results=1)
keyword_top = bm25_search(test_query, n_results=1)
hybrid_top = hybrid_search(test_query, n_results=1)

comparison = pd.DataFrame({
    "Method": ["Semantic only", "Keyword (BM25) only", "Hybrid (RRF)"],
    "Top Result": [
        semantic_top[0]["text"][:70] + "..." if semantic_top else "(no result)",
        keyword_top[0]["text"][:70] + "..." if keyword_top else "(no result)",
        hybrid_top[0]["text"][:70] + "..." if hybrid_top else "(no result)",
    ]
})
comparison

,Method,Top Result
0,Semantic only,Product Catalog Notes\n\nThe wireless noise-ca...
1,Keyword (BM25) only,The standing desk converter is listed under pr...
2,Hybrid (RRF),The standing desk converter is listed under pr...


In [8]:
def distance_to_similarity(distance):
    return 1 - distance

def retrieve_with_threshold(query, n_results=5, similarity_threshold=0.3):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    filtered = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = distance_to_similarity(distance)
        if similarity >= similarity_threshold:
            filtered.append({"text": doc, "source": meta["source"], "similarity": round(similarity, 4)})
    return filtered

# An out-of-scope query - our documents know nothing about cooking
empty_test = retrieve_with_threshold("What's the best recipe for chocolate cake?")
print("Chunks passing the threshold:", len(empty_test))
for r in empty_test:
    print(r)

Chunks passing the threshold: 0


In [9]:
KNOWLEDGE_BASE_TOPICS = sorted({meta["source"] for meta in all_chunk_meta})

def smart_retrieve(query, n_results=4, similarity_threshold=0.3):
    # Step 1: try semantic search with a quality threshold
    primary = retrieve_with_threshold(query, n_results=n_results, similarity_threshold=similarity_threshold)
    if primary:
        return {"status": "ok", "method": "semantic", "chunks": primary}

    # Step 2: fall back to hybrid search (catches exact keyword matches semantic search missed)
    hybrid_results = hybrid_search(query, n_results=n_results)
    if hybrid_results:
        return {"status": "ok", "method": "hybrid_fallback", "chunks": hybrid_results}

    # Step 3: nothing relevant found anywhere - fail honestly instead of guessing
    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            f"This knowledge base currently covers: {', '.join(KNOWLEDGE_BASE_TOPICS)}."
        )
    }

# Test 1: a query that should succeed through normal semantic search
result = smart_retrieve("What is deep learning?")
print("Test 1 ->", result["status"], "|", result["method"])

# Test 2: an exact-code query that may need the hybrid fallback
result = smart_retrieve("Tell me about SKU-4521")
print("Test 2 ->", result["status"], "|", result["method"])

# Test 3: a genuinely out-of-scope query
result = smart_retrieve("What's the best recipe for chocolate cake?")
print("Test 3 ->", result["status"], "|", result.get("message"))

Test 1 -> ok | semantic
Test 2 -> ok | semantic
Test 3 -> ok | None


In [10]:
def format_context_for_llm(query, chunks):
    context_text = "\n\n".join(f"[Source: {c['source']}]\n{c['text']}" for c in chunks)
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

def safe_rag_prompt(query):
    retrieval = smart_retrieve(query)

    if retrieval["status"] == "empty":
        # No LLM call needed - we already know the honest answer
        return retrieval["message"]

    prompt = format_context_for_llm(query, retrieval["chunks"])
    return f"[Would send this prompt to the LLM, retrieved via '{retrieval['method']}']\n\n{prompt}"

print(safe_rag_prompt("What is SKU-4521?"))
print("\n" + "=" * 70 + "\n")
print(safe_rag_prompt("What's the best recipe for chocolate cake?"))

[Would send this prompt to the LLM, retrieved via 'semantic']

Context:
[Source: product_catalog.txt]
Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.

Question: What is SKU-4521?

Answer:


[Would send this prompt to the LLM, retrieved via 'hybrid_fallback']

Context:
[Source: product_catalog.txt]
Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.

[Source: ai_overview.txt]
Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence.

[Source: product_catalog.txt]
Product Catalog Notes

The wireless noise-cancelling headp

## Practice Exercises — Completed Solutions

The cells below implement all three exercises. Run them after the main notebook cells above.

### Exercise 1 — Change RRF constant from 60 to 10
RRF gives more influence to rank position differences when `k` is smaller. Compare the top results for both values; exact ordering depends on the search results.

In [11]:
test_query = "What is SKU-4521?"

rrf_k60 = hybrid_search(test_query, n_results=5, k=60)
rrf_k10 = hybrid_search(test_query, n_results=5, k=10)

print("RRF with k=60")
for rank, item in enumerate(rrf_k60, start=1):
    print(f"{rank}. [{item['rrf_score']}] ({item['source']}) {item['text'][:100]}")

print("\\nRRF with k=10")
for rank, item in enumerate(rrf_k10, start=1):
    print(f"{rank}. [{item['rrf_score']}] ({item['source']}) {item['text'][:100]}")

RRF with k=60
1. [0.03227] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
da
2. [0.03202] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-45
3. [0.032] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
t
4. [0.03151] (ai_overview.txt) Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardc
5. [0.03009] (ai_overview.txt) Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer scien
\nRRF with k=10
1. [0.16783] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
da
2. [0.16234] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-45
3. [0.16026] (product_catalo

### Exercise 2 — Add a fourth document with a new product code


In [12]:
new_source = "accessories_catalog.txt"
new_document = """Accessories Catalog

The ergonomic wireless keyboard is listed under product code SKU-9912.
It includes adjustable typing angles and a rechargeable battery.
"""

new_chunks = splitter.split_text(new_document)
new_ids = [f"{new_source}_chunk_{i}" for i in range(len(new_chunks))]
new_meta = [{"source": new_source, "chunk_index": i} for i in range(len(new_chunks))]

# Add the document to ChromaDB and the lists used by BM25/hybrid search.
collection.add(documents=new_chunks, metadatas=new_meta, ids=new_ids)
all_chunks.extend(new_chunks)
all_chunk_ids.extend(new_ids)
all_chunk_meta.extend(new_meta)

# Rebuild BM25 because its corpus has changed.
tokenized_chunks = [tokenize(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

# Refresh the displayed knowledge-base topics.
KNOWLEDGE_BASE_TOPICS = sorted({meta["source"] for meta in all_chunk_meta})

result = smart_retrieve("Tell me about SKU-9912", similarity_threshold=0.99)
print("Status:", result["status"])
print("Method:", result["method"])
for item in result["chunks"]:
    print(f"[{item.get('similarity', item.get('rrf_score'))}] ({item['source']}) {item['text']}")

Status: ok
Method: hybrid_fallback
[0.03252] (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
[0.03202] (product_catalog.txt) The standing desk converter is listed under product code SKU-7788 and ships within three business
days.
[0.032] (accessories_catalog.txt) Accessories Catalog

The ergonomic wireless keyboard is listed under product code SKU-9912.
It includes adjustable typing angles and a rechargeable battery.
[0.0315] (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.


### Exercise 3 — Add a stopword-removal fallback

In [13]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "what", "who", "where",
    "when", "why", "how", "do", "does", "did", "tell", "me", "about",
    "please", "for", "of", "to", "in", "on", "and", "or", "with", "it"
}

def remove_stopwords(query):
    return " ".join(token for token in tokenize(query) if token not in STOPWORDS)

def smart_retrieve_with_stopwords(query, n_results=4, similarity_threshold=0.3):
    # Step 1: semantic search with a quality threshold.
    primary = retrieve_with_threshold(
        query, n_results=n_results, similarity_threshold=similarity_threshold
    )
    if primary:
        return {"status": "ok", "method": "semantic", "chunks": primary}

    # Step 2: hybrid search with the original query.
    hybrid_results = hybrid_search(query, n_results=n_results)
    if hybrid_results:
        return {"status": "ok", "method": "hybrid_fallback", "chunks": hybrid_results}

    # Step 3: remove stopwords and try hybrid search again.
    cleaned_query = remove_stopwords(query)
    if cleaned_query and cleaned_query.lower() != query.lower():
        cleaned_results = hybrid_search(cleaned_query, n_results=n_results)
        if cleaned_results:
            return {
                "status": "ok",
                "method": "stopword_hybrid_fallback",
                "chunks": cleaned_results,
                "cleaned_query": cleaned_query,
            }

    # Step 4: be honest when no retrieval strategy finds a match.
    return {
        "status": "empty",
        "method": None,
        "chunks": [],
        "message": (
            "I don't know based on the provided documents. "
            f"This knowledge base currently covers: {', '.join(KNOWLEDGE_BASE_TOPICS)}."
        ),
    }

for query in [
    "Can you please tell me about SKU-9912?",
    "What's the best recipe for chocolate cake?"
]:
    result = smart_retrieve_with_stopwords(query)
    print(f"Query: {query}")
    print("Status:", result["status"], "| Method:", result["method"])
    if result["status"] == "empty":
        print(result["message"])
    else:
        for chunk in result["chunks"]:
            print(f"  ({chunk['source']}) {chunk['text']}")
    print()

Query: Can you please tell me about SKU-9912?
Status: ok | Method: semantic
  (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
  (accessories_catalog.txt) Accessories Catalog

The ergonomic wireless keyboard is listed under product code SKU-9912.
It includes adjustable typing angles and a rechargeable battery.
  (product_catalog.txt) Product Catalog Notes

The wireless noise-cancelling headphones are listed under product code SKU-4521 and are currently
our best-selling audio accessory.

Query: What's the best recipe for chocolate cake?
Status: ok | Method: hybrid_fallback
  (product_catalog.txt) Return requests for any product must reference the exact SKU code so our warehouse team can locate
the correct item quickly.
  (ai_overview.txt) Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks 